## <center>Test technique : Prédiction du vainqueur d'un match de tennis - Siwar ABBES</center> 
------------------------------------------------------------------------------------------------------------------


1. [Importation des librairies](#Importation)
2. [Exploration des données](#EDA)
3. [Feature Engineering](#engineering)
4. [Classification](#Classification)
5. [Evaluation](#Evaluation)

#### Le but est de réaliser un modèle prédisant le vainqueur d’un match de tennis et d’en tirer de la valeur exploitable par un client.

Le livrable est un dossier contenant : 

- des scripts python exécutables et exécutés
- Un README.md présentant les instructions pour exécuter le code et les résultats dans les grandes lignes
- Un dossier avec des output (images et visualisations)
- Tout autre fichier qui permet d’exécuter le code et montrer les résultats

On souhaite avoir :

- Une exploration des données
- L'entra√Ænement de modèlesde prédiction
- Une analyse des performances
- Une présentation d'un cas d'utilisation
- Des pistes pour poursuivre le PoC
- Des script permettant la mise en production du code
chacune de ces parties fait l'objet d'une section dans le README.md

Conditions de passation :

A la maison
Cet exercice doit être réalisé en 4h maximum
Dans des conditions normales de travail (accès à internet, etc.)
Critères d'évaluation :

- Qualité de la méthode data science (la performance du modèle n’est pas le critère d’évaluation principal de l’exercice) ;
- Qualité de la compréhension du problème client, des données et des solutions envisagées
- Qualité de présentation des résultats
- Qualité du code: structuration, modularité, documentation
Important :

Merci de noter votre NOM Prénom sur le nom de fichier que vous nous transmettez.
Merci également de ne pas diffuser vos résultats publiquement (kaggle, github, gitlab, …). »

<a id='Importation'></a>
### 1. Importation des librairies

In [1]:
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('ticks')
sns.set(rc={'figure.figsize':(8,6)})
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import json

<a id='EDA'></a>
### 2. Exploration des données

In [4]:
data = pd.read_csv("data/ATP_tweaked.csv", sep = ';')
print('La taille des données est : ', data.shape)
data.head()

La taille des données est :  (12310, 49)


,best_of,p2_1stIn,p2_1stWon,p2_2ndWon,p2_SvGms,p2_ace,p2_bpFaced,p2_bpSaved,p2_df,p2_svpt,...,p1_entry,p1_hand,p1_ht,p1_id,p1_ioc,p1_name,p1_rank,p1_rank_points,p1_seed,p1_won
0,3,66.0,47.0,26.0,15.0,9.0,9.0,8.0,6.0,112.0,...,NaN,R,188.0,104542,FRA,Jo Wilfried Tsonga,19.0,1645.0,NaN,0
1,3,45.0,33.0,21.0,13.0,6.0,9.0,6.0,5.0,87.0,...,NaN,R,NaN,105526,GER,Jan Lennard Struff,89.0,650.0,NaN,1
2,5,54.0,38.0,21.0,18.0,3.0,4.0,0.0,3.0,92.0,...,NaN,R,183.0,104527,SUI,Stanislas Wawrinka,5.0,5710.0,5.0,1
3,3,46.0,35.0,17.0,14.0,2.0,2.0,0.0,3.0,72.0,...,NaN,R,180.0,104655,URU,Pablo Cuevas,23.0,1447.0,6.0,0
4,3,42.0,33.0,13.0,11.0,10.0,1.0,0.0,1.0,68.0,...,Q,R,185.0,104433,CAN,Frank Dancevic,156.0,326.0,NaN,0


In [7]:
data.drop_duplicates(inplace=True)
data.shape

(12310, 49)

In [36]:
data.columns

Index(['best_of', 'p2_1stIn', 'p2_1stWon', 'p2_2ndWon', 'p2_SvGms', 'p2_ace',
       'p2_bpFaced', 'p2_bpSaved', 'p2_df', 'p2_svpt', 'p2_age', 'p2_entry',
       'p2_hand', 'p2_ht', 'p2_id', 'p2_ioc', 'p2_name', 'p2_rank',
       'p2_rank_points', 'p2_seed', 'match_num', 'minutes', 'round', 'score',
       'surface', 'tourney_date', 'tourney_id', 'tourney_level',
       'tourney_name', 'p1_1stIn', 'p1_1stWon', 'p1_2ndWon', 'p1_SvGms',
       'p1_ace', 'p1_bpFaced', 'p1_bpSaved', 'p1_df', 'p1_svpt', 'p1_age',
       'p1_entry', 'p1_hand', 'p1_ht', 'p1_id', 'p1_ioc', 'p1_name', 'p1_rank',
       'p1_rank_points', 'p1_seed', 'p1_won'],
      dtype='object')

The data set contains the details about all the ATP matches played since 1968. The match
statistics are available for matches since 1991.


General
- tourneyid - tournamentid
- tourneyname - tournamentname
- surface - surface in which the match is played
- draw_size - the size of the draw
- tourney_level - tournament level
    - 'G' = Grand Slams
    - 'M' = Masters 1000s
    - 'A' = other tour-level events
    - 'C' = Challengers
    - 'S' = Satellites/ITFs
    - 'F' = Tour finals and other season-ending events
    - 'D' = Davis Cup
- tourney_date - starting date of the tournament
- match num - match number in a certain tournament
Players
- [prefixes : P1_ - info about player 1, P2_ - info about player 2]
- id - player id
- seed - the seed of the player in that tournament
- entry - How did the player enter the tournaments?
    - WC - Wildcard
    - Q - Qualifier
    - LL - Lucky loser
    - PR - Protected ranking
    - SE - Special Exempt
    - ALT - Alternate player
- name - player name
- hand - hand of the player, right or left
- ht - the height of the player
- IOC - the country of origin
- age - age of the player
- score - final score in the match
- best_of - the maximum number of sets played
- round - the round in the tournament a match belongs to
- minutes - duration of the match in minutes
- ace - number of aces in the match
- df - double faults
- svpt - serve percent
- 1stin - first serve in percent
- 1stWon - first serve winning percent
- 2ndWon - second serve winning percent
- SvGms - number of games played on serve (So, the maximum difference between
wSvGms and lSvGms will be 1)
- bpSaved - breakpoints save
- bpFaced - breakpoints faced
- p1_won : first player won

**score**

In [20]:
data['score'][0]

'4-6 6-3 6-4'

In [31]:
data['score'][0].split()]

['4-6', '6-3', '6-4']

In [34]:
nb_sets = [len(data["score"][0].split())]
games_during_all_matchs = [g.split('-') for g in data["score"][0].split()]
games_list = []
for s in games_during_all_matchs:
    games = 0 
    for g in s :
        try:
            games += int(g[0]) + int(g[1]) 
        except:
            continue
    games_list.append(games)
print(f'nb of sets {nb_sets},\ngames during match{games_during_all_matchs}')

nb of sets [3],
games during match[['4', '6'], ['6', '3'], ['6', '4']]


In [15]:
data['score'][6]

'6-2 6-0'

In [18]:
[len(data["score"][6].split())]

[2]

In [8]:
features = {c: data[c].nunique() for c in data.columns}
features_with_nb_categories ={k:v for k, v in sorted(features.items(), key=lambda item: item[1], reverse=True)} 
features_with_nb_categories

{'score': 3729,
 'p1_rank_points': 1987,
 'p2_rank_points': 1964,
 'p2_age': 1928,
 'p1_age': 1914,
 'p1_rank': 716,
 'p2_rank': 683,
 'p2_id': 657,
 'p2_name': 657,
 'p1_id': 628,
 'p1_name': 628,
 'tourney_id': 579,
 'tourney_name': 374,
 'match_num': 343,
 'minutes': 284,
 'p2_svpt': 213,
 'p1_svpt': 211,
 'tourney_date': 179,
 'p1_1stIn': 146,
 'p2_1stIn': 142,
 'p1_1stWon': 114,
 'p2_1stWon': 110,
 'p2_ioc': 88,
 'p1_ioc': 86,
 'p2_2ndWon': 54,
 'p1_2ndWon': 54,
 'p1_ace': 52,
 'p2_ace': 51,
 'p2_SvGms': 40,
 'p1_SvGms': 40,
 'p2_seed': 33,
 'p1_seed': 33,
 'p2_bpFaced': 31,
 'p1_bpFaced': 30,
 'p2_bpSaved': 25,
 'p1_bpSaved': 24,
 'p2_df': 21,
 'p1_df': 20,
 'p2_ht': 19,
 'p1_ht': 19,
 'round': 9,
 'p2_entry': 7,
 'p1_entry': 6,
 'surface': 5,
 'tourney_level': 5,
 'p2_hand': 3,
 'p1_hand': 3,
 'best_of': 2,
 'p1_won': 2}

In [43]:
data['p1_entry'].unique()

array([nan, 'Q', 'WC', 'PR', 'LL', 'SE', 'ALT'], dtype=object)

In [ ]:
# get some info about the features and the missing values
#data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum(axis=0) * 100 / len(data)

In [ ]:
sk_kt_study_list = [n for n in numerical_variables if n not in ["p1_ht", "p1_age", "p1_rank", "p1_rank_points", "p1_seed", "p2_ht", "p2_age", "p2_rank", "p2_rank_points", "p2_seed"]]

--------------------------------------------------------------------------------------------

### 2.1 Explorer la variable objective (the target)
Selon le jeu de données, nous pouvons évidemment conclure que la variable objective (Target object) à prédire est p1_won; si le premier joueur gagnerait la partie.

In [ ]:
# Explore the target
print(" Le nombre de valeurs nulles pour la target:", data["p1_won"].isnull().sum())


print("\n Le nombre de victoires pour la target:", data["p1_won"].value_counts()[1])

print("\n Le pourcentage de victoires pour la target:", data["p1_won"].value_counts()[1] / data.shape[0] * 100, '%')

In [ ]:
plt.figure(figsize=(6,3))
ax= sns.countplot(x='p1_won', data=data)

Conclusion : 
On est face à un problème supervisé de <b> classification binaire avec des classes équilibrées. </b> <br>
Les métriques qu'il faudrait donc utiliser pour juger la performance du modèle sont :

- Accuracy
- Precision
- Recall
- F1-score : une mesure qui permet de donner une valeur équilibrée entre la précision et le recall

La première chose à faire est de distinguer les différents types de variables présentes sur le dataset c'est à dire voir s'ils correspondent à des variables catégoriques, numériques ou ordinales.

- **Les variables catégoriques** prennent en effet des valeurs dans un ensemble finit de valeur, la question qui reste à savoir est le cardinal de cette ensemble. **Nous supposerons que la valeur limite à partir de laquelle nous considérerons la variable comme non catégorique est 10**

- **Les variables numériques** sont ceux qui prennent des valeurs entières ou réelles et ils sont faciles à identifier. Il faut faire attention à un point certaines variables prennent des valeurs numériques mais ne correspondent pas à des variables numériques comme par exemple les Ids ou les valeurs ordinales qu'il faudrait donc distinguer 

- **Les valeurs ordinales** sont des variables dont les valeurs sont définies par une relation d’ordre entre les catégories possibles et sont souvent des valeurs entières.

**Observations :** Nous pouvons constater sur les variables présélectionnées comme étant numérique que certaines sont plutôt ordinales dans la mesure où il représente une certaine relation d'ordre. Les colonnes en question sont les suivantes:
    
- La taille :
    - p2_ht
    - p1_ht
- L'age :
    - p2_age
    - p1_age
- Le rang des joueurs:
    - p2_rank
    - p1_rank
- Le nombre de point gagnés sur le rang de l'ATP:
    - p2_rank_points
    - p1_rank_points
    
Un autre point intéressant que j'ai observé est que la variable **tourney_date** est indiquée sous forme numérique alors que ce n'est pas le format adéquat pour représenter une date et qui nécessite donc un prétraitement.

### 2.2 Variables catégoriques

In [ ]:
data.best_of.unique()

#### tourney_level : tournament level


In [ ]:
#ax = sns.countplot(x='tourney_level', data=data)
#sns.barplot(y='p1_won', x='tourney_level', data=data) 
ax = sns.countplot(x="tourney_level", hue="p1_won", data=data, palette=sns.color_palette(
    "Set2", n_colors=2))
ax.set_title("Impact of tournament level on winning ")

    - 'G' = Grand Slams
    - 'M' = Masters 1000s
    - 'A' = other tour-level events
    - 'C' = Challengers
    - 'S' = Satellites/ITFs
    - 'F' = Tour finals and other season-ending events
    - 'D' = Davis Cup

As we can notice here, there are not data for S and C tournement level

In [ ]:
data.tourney_level.unique()

#### Surface Type

In [ ]:
ax = sns.countplot(x="surface", hue="p1_won", data=data, palette=sns.color_palette(
    "Set2", n_colors=2))
ax.set_title("Impact of surface type on tennis result ")

#### Entry level
How did the player enter the tournaments?


In [ ]:
ax = sns.countplot(x="p1_entry", hue="p1_won", data=data, palette=sns.color_palette("Set2", n_colors=2))
ax.set_title("Impact of p1 entry level on tennis result ")
plt.show()

- WC - Wildcard
- Q - Qualifier
- LL - Lucky loser
- PR - Protected ranking
- SE - Special Exempt
- ALT - Alternate player

### Variables numériques et ordinales

In [ ]:
plt.figure(figsize=(15,6))
sns.distplot(data['p1_ht'], label='Height distribution')
plt.axvline(x=data['p1_ht'].mean(), color='red', linestyle='--', label='Mean Height')
plt.axvline(x=data['p1_ht'].median(), color='black', linestyle='--', label='Median Height')
plt.title('Distribution of the Height of the player 1')
plt.legend()
plt.show()

### Date

In [ ]:
data.tourney_date.min(), data.tourney_date.max() 

In [ ]:
type(data.tourney_date[0])

In [ ]:
plt.figure(figsize=(15,6))
sns.distplot(data['p1_age'], label='Age distribution')
plt.axvline(x=data['p1_age'].mean(), color='red', linestyle='--', label='Mean age')
plt.axvline(x=data['p1_age'].median(), color='black', linestyle='--', label='Median age')
plt.title('Distribution of the age of the player 1')
plt.legend()
plt.show()

<a id = 'engineering'></a>
### 3. Feature Engineering

**Observations :** Nous pouvons constater sur les variables présélectionnées comme étant numérique que certaines sont plutôt ordinales dans la mesure où il représente une certaine relation d'ordre. Les colonnes en question sont les suivantes:
    
- La taille :
    - p2_ht
    - p1_ht
- L'age :
    - p2_age
    - p1_age
- Le rang des joueurs:
    - p2_rank
    - p1_rank
- Le nombre de point gagnés sur le rang de l'ATP:
    - p2_rank_points
    - p1_rank_points


**Dealing with Nan values:**

In [ ]:
data.p2_seed.unique()

In [ ]:
data.p1_entry.unique()

In [ ]:
class PreProcessing:
    ''' Deal with nan values, drop useless columns'''
    
    def __init__(self, data: pd.DataFrame) -> None:
        self.data = data.copy()
    
    def get_columns_with_nan_values(self) -> dict[str, str]:
        """
        Find columns that contain nan values.
        """
        features_with_nan_values : dict[str, float]={}
        nb_lines = self.data.shape[0]
        for c in self.data.columns :
            if self.data[c].count() < nb_lines:
                features_with_nan_values[c] = round((1 - self.data[c].count()/self.data.shape[0]) * 100, 2)
              
        return {k: str(v)+'%' for k,v in sorted(features_with_nan_values.items(), key=lambda item: item[1], reverse=True)}
                
 
    def _fill_nan_values(self, cols_median:list[str], cols_categorical:list[str]) -> None:
        """
        Fill nan values
        """
        features_with_nan_values = self.get_columns_with_nan_values()
        print(f'\nFeatures with Nan values : {json.dumps(features_with_nan_values, indent = 4)}')
        print(f'\nFeatures to fill nan values by median : {cols_median}')
        print(f'\nFeatures to fill nan values by a new category unknown : {cols_categorical}')
              
        for c in cols_median:
            self.data[c] = self.data[c].fillna((self.data[c].median()))
                        
        for c in cols_categorical:
            # fill nan values by a new category unknown(unk)
            self.data[c] = self.data[c].astype('str').replace("nan", "unk") 
        
        ## drop the value of the hand column
        self.data.p1_hand.fillna('U')
        self.data.p2_hand.fillna('U')
        self.data.reset_index(inplace=True, drop=True)
        
        features_with_nan_values = self.get_columns_with_nan_values()
        print(f'\nFeatures with Nan values after pre-processing: {features_with_nan_values}')
            
        return None
    
    def preprocess(self, cols_to_drop: list[str], cols_median:list[str], cols_categorical:list[str]) -> pd.DataFrame:
        # drop duplicates
        print(f'Features to drop: {features_to_drop}')
        self.data.drop_duplicates(inplace=True)
        # drop useless cols
        self.data.drop(columns=cols_to_drop, axis=1)
        

        # fill nan values
        self._fill_nan_values(cols_median, cols_categorical)
        return self.data
        


In [ ]:
features_to_drop = ["tourney_name", "tourney_id", "p2_name", "p1_name", "p2_id", "p1_id"]
features_to_fill_by_median_per_player = [[f"p{p}_1stIn", f"p{p}_1stWon", 
              f"p{p}_2ndWon", f"p{p}_SvGms", 
              f"p{p}_ace", f"p{p}_bpFaced", 
              f"p{p}_bpSaved", f"p{p}_df", 
              f"p{p}_svpt", f"p{p}_rank", 
              f"p{p}_rank_points", f"p{p}_age", 
              f"p{p}_ht"] for p in range(1,3)]
features_to_fill_by_median = features_to_fill_by_median_per_player[0] + features_to_fill_by_median_per_player[1] + ["minutes"]

features_to_fill_by_new_category = [f"p1_seed", f"p1_entry", f"p2_seed", f"p2_entry"]

features_to_fill_by_median

In [ ]:

pre_processing = PreProcessing(data)

data = pre_processing.preprocess(features_to_drop, features_to_fill_by_median, features_to_fill_by_new_category)


In [ ]:
df_numerical = data.drop(features_to_fill_by_new_category, axis=1)
numerical_features = df_numerical.describe().columns
sk_kt_study_list = [n for n in numerical_features if n not in ["p1_ht", "p1_age", "p1_rank", "p1_rank_points", "p1_seed", "p2_ht", "p2_age", "p2_rank", "p2_rank_points", "p2_seed"]]        


In [ ]:
class FeatureEngineering:
    
    
    def __init__(self, data: pd.DataFrame) -> None:
        self.data = data.copy()
        
    def _split_date(self)-> None:
        self.data['tourney_date'] = self.data['tourney_date'].map(lambda x : str(x).split(" ")[0])
        self.data['tourney_date'] = pd.to_datetime(self.data['tourney_date'])
        self.data['day_of_week'] = self.data['tourney_date'].apply(lambda val: val.day_name())
        self.data['month'] = self.data['tourney_date'].apply(lambda val: val.month_name())
        #maybe we need to drop tourney_date because it's not a time series problem
        
        
    def _importance_tourney(self) -> None:
        """
        Set A new column which represent the importance of the tourney.
        """
        importance_dict = {"G":6, "M": 5, "A": 4, "C": 4, "S": 3, "F" : 2, "D": 1}
        importance = [importance_dict[k] for k in self.data["tourney_level"].tolist()]
        self.data["tourney_importance"] = importance
        
    def _sets_per_match(self) -> None:
        """
        Get the number of set per match.
        """
        self.data["nbr_sets"] = [len(match.split()) for match in self.data["score"]]
    
    def _game_per_match(self) -> None:
        """
        Get the number of game played during a match.
        """
        games_during_all_matchs = [[g.split('-') for g in match.split()] for match in self.data["score"]]
        
        games_list = []
        for s in games_during_all_matchs:
            games = 0 
            for g in s :
                try:
                    games += int(g[0]) + int(g[1]) 
                except:
                    continue
            games_list.append(games)
            
        self.data['games_per_match'] = games_list
        self.data.drop(columns=["score"], inplace=True)
        
    def _process_categorical_data(self, categorical_cols: list[str]) -> None:
        """
        Apply get dummies for categorical data.
        """
        self.data = pd.get_dummies(self.data, columns=categorical_cols)
        
    def _log_transform(self, sk_kt_cols: list[str]) -> None:
        """
        Apply log function to normalize numerical columns that need to be transformed
        """
        for col in sk_kt_cols:
            try:
                self.data[f'log_{col}'] = np.log1p(self.data[col])
            except:
                sk_kt_cols.remove(col)
        self.data.drop(columns=sk_kt_cols, inplace=True)
    
    def _transform_seed_as_categorical(self, categorical_cols: list[str])-> None:
        """
        Consider seed as a categorical variable.
        """
        for p in [1, 2]:    
            self.data[f"p{p}_seed"] = self.data[f"p{p}_seed"].astype(str)
            categorical_cols.append(f"p{p}_seed")
            
    
    
    def transform(self, categorical_cols: list[str], sk_kt_cols: list[str]) -> pd.DataFrame:
        """
        Apply the feature engineering pipeline.
        """
        # Add date features : 
        self._split_date()
        # Add importance tourney feature:
        self._importance_tourney()
        # Add number of set per match:
        self._sets_per_match()
        # Add the number of game pet match:
        self._game_per_match()
        # Scale numerical data in order to deal with outliers problem:
        self._log_transform(sk_kt_cols)
        # Transform seed column as categorical variable
        self._transform_seed_as_categorical(categorical_cols)
        # create a dummies variables for categorical data
        self._process_categorical_data(categorical_cols)
        return self.data

In [ ]:

pre_processing = PreProcessing(data)
data = pre_processing.preprocess(cols_to_drop, cols_median, cols_categorical)

feature_engineering = FeatureEngineering(data)
data = feature_engineering.transform(cols_categorical, sk_kt_study_list)

In [ ]:
data.head()

In [ ]:
data.p1_name.unique()

In [35]:
{c: data[c].nunique() for c in data.columns}

{'best_of': 2,
 'p2_1stIn': 142,
 'p2_1stWon': 110,
 'p2_2ndWon': 54,
 'p2_SvGms': 40,
 'p2_ace': 51,
 'p2_bpFaced': 31,
 'p2_bpSaved': 25,
 'p2_df': 21,
 'p2_svpt': 213,
 'p2_age': 1928,
 'p2_entry': 7,
 'p2_hand': 3,
 'p2_ht': 19,
 'p2_id': 657,
 'p2_ioc': 88,
 'p2_name': 657,
 'p2_rank': 683,
 'p2_rank_points': 1964,
 'p2_seed': 33,
 'match_num': 343,
 'minutes': 284,
 'round': 9,
 'score': 3729,
 'surface': 5,
 'tourney_date': 179,
 'tourney_id': 579,
 'tourney_level': 5,
 'tourney_name': 374,
 'p1_1stIn': 146,
 'p1_1stWon': 114,
 'p1_2ndWon': 54,
 'p1_SvGms': 40,
 'p1_ace': 52,
 'p1_bpFaced': 30,
 'p1_bpSaved': 24,
 'p1_df': 20,
 'p1_svpt': 211,
 'p1_age': 1914,
 'p1_entry': 6,
 'p1_hand': 3,
 'p1_ht': 19,
 'p1_id': 628,
 'p1_ioc': 86,
 'p1_name': 628,
 'p1_rank': 716,
 'p1_rank_points': 1987,
 'p1_seed': 33,
 'p1_won': 2}

In [ ]:
cats_day = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ax = data.groupby('day_of_week').agg({'p1_won':'mean'}).reindex(cats_day).plot(figsize=(12,6))
ticks = list(range(0, 7, 1)) 
labels = "Mon Tues Weds Thurs Fri Sat Sun".split()
plt.xticks(ticks, labels)
plt.title('Winnings of player 1 by day of week');

In [ ]:
ax = data.groupby('month').agg({'p1_won': 'mean'}).plot(figsize=(12,6))
ticks = list(range(1, 13, 1)) # points on the x axis where labels appear
labels = 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
plt.xticks(ticks,labels)
plt.title('Number of p1 winnings per month');